## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
#include <bits/stdc++.h>
using namespace std;
class PermutationOperator {
private:
    int sizeN, keyX, keyY;
    int segmentLen;
    vector<int> currentSeq;
    vector<int> whereIs;
    vector<int> commandList;
public:
    void solve() {
        loadData();
        calcSegmentLen();
        if (!alignBasePart()) {
            cout << "-1\n";
            return;
        }
        if (!makeIdentity()) {
            cout << "-1\n";
            return;
        }
        outputCommands();
    }
private:
    void loadData() {
        ios::sync_with_stdio(false);
        cin.tie(nullptr);
        cin >> sizeN >> keyX >> keyY;
        currentSeq.resize(sizeN);
        for (int idx = 0; idx < sizeN; ++idx) {
            cin >> currentSeq[idx];
        }
        whereIs.resize(sizeN);
        refreshIndex();
    }
    void refreshIndex() {
        for (int idx = 0; idx < sizeN; ++idx) {
            whereIs[currentSeq[idx]] = idx;
        }
    }
    void calcSegmentLen() {
        segmentLen = (keyX - keyY + sizeN) % sizeN;
        segmentLen &= -segmentLen;
        if (segmentLen == 0) {
            segmentLen = sizeN;
        }
    }
    void useKeyExchange() {
        commandList.push_back(0);
        for (int idx = 0; idx < sizeN; ++idx) {
            if (currentSeq[idx] == keyX) {
                currentSeq[idx] = keyY;
            } else if (currentSeq[idx] == keyY) {
                currentSeq[idx] = keyX;
            }
        }
        refreshIndex();
    }
    void useShift(int offset) {
        offset %= sizeN;
        if (offset < 0) {
            offset += sizeN;
        }
        if (offset == 0) {
            return;
        }
        commandList.push_back(offset);
        for (int idx = 0; idx < sizeN; ++idx) {
            currentSeq[idx] = (currentSeq[idx] + offset) % sizeN;
        }
        refreshIndex();
    }
    void useXor(int mask) {
        if (mask == 0) {
            return;
        }
        commandList.push_back(-mask);
        for (int idx = 0; idx < sizeN; ++idx) {
            currentSeq[idx] ^= mask;
        }
        refreshIndex();
    }
    pair<int, int> locatePair(int leftVal, int rightVal) {
        int gap = (rightVal - leftVal + sizeN - segmentLen + sizeN) % sizeN;
        int firstPos = 0;
        int secondPos = 0;
        int stride = sizeN / 2;
        while (stride >= 2 * segmentLen) {
            if (gap >= stride) {
                gap -= stride;
                secondPos += stride / 2;
            } else {
                firstPos += stride / 2;
            }
            stride >>= 1;
        }
        firstPos += sizeN / 2;
        firstPos += (leftVal & (segmentLen - 1));
        secondPos += (leftVal & (segmentLen - 1));
        return {firstPos, secondPos};
    }
    void exchangeValuePair(int leftVal, int rightVal) {
        int sideLeft = (leftVal / segmentLen) % 2;
        int sideRight = (rightVal / segmentLen) % 2;
        if (sideLeft == sideRight) {
            int bridgeVal;
            if (sideLeft == 0) {
                bridgeVal = (leftVal & (segmentLen - 1)) + segmentLen;
            } else {
                bridgeVal = (leftVal & (segmentLen - 1));
            }
            exchangeValuePair(leftVal, bridgeVal);
            exchangeValuePair(rightVal, bridgeVal);
            exchangeValuePair(leftVal, bridgeVal);
            return;
        }
        auto [baseLeftPos, baseRightPos] = locatePair(keyX, keyY);
        auto [needLeftPos, needRightPos] = locatePair(leftVal, rightVal);
        useShift((needLeftPos - leftVal + sizeN) % sizeN);
        useXor(needLeftPos ^ baseLeftPos);
        useShift((keyX - baseLeftPos + sizeN) % sizeN);
        useKeyExchange();
        useShift((baseLeftPos - keyX + sizeN) % sizeN);
        useXor(needLeftPos ^ baseLeftPos);
        useShift((leftVal - needLeftPos + sizeN) % sizeN);
    }
    pair<bool, vector<int>> decomposeOrder(const vector<int>& source, int width) {
        vector<bool> seen(width, false);
        for (int idx = 0; idx < width; ++idx) {
            if (source[idx] >= width) {
                return {false, {}};
            }
            seen[source[idx]] = true;
        }
        for (int idx = 0; idx < width; ++idx) {
            if (!seen[idx]) {
                return {false, {}};
            }
        }
        if (width == 1) {
            return {true, {}};
        }
        int halfWidth = width / 2;
        vector<int> evenPart(halfWidth);
        vector<int> oddPart(halfWidth);
        for (int idx = 0; idx < halfWidth; ++idx) {
            evenPart[idx] = source[idx * 2] / 2;
            oddPart[idx] = source[idx * 2 + 1] / 2;
        }
        auto [evenOk, evenOps] = decomposeOrder(evenPart, halfWidth);
        auto [oddOk, oddOps] = decomposeOrder(oddPart, halfWidth);
        if (!evenOk || !oddOk) {
            return {false, {}};
        }
        vector<int> rawOps;
        if (source[0] & 1) {
            if (width == 2) {
                rawOps.push_back(1);
            } else {
                rawOps.push_back(-1);
            }
        }
        int evenMask = 0;
        for (int item : evenOps) {
            if (item > 0) {
                rawOps.push_back(-1);
                rawOps.push_back(1);
            } else {
                rawOps.push_back(item * 2);
                evenMask ^= (-item) * 2;
            }
        }
        if (evenMask != 0) {
            rawOps.push_back(-evenMask);
        }
        int oddMask = 0;
        for (int item : oddOps) {
            if (item > 0) {
                rawOps.push_back(1);
                rawOps.push_back(-1);
            } else {
                rawOps.push_back(item * 2);
                oddMask ^= (-item) * 2;
            }
        }
        if ((oddMask & halfWidth) != (evenMask & halfWidth)) {
            return {false, {}};
        }
        if (evenMask >= halfWidth) {
            evenMask -= halfWidth;
        }
        if (oddMask >= halfWidth) {
            oddMask -= halfWidth;
        }
        if (evenMask != oddMask) {
            return {false, {}};
        }
        vector<int> compactOps;
        for (int item : rawOps) {
            if (compactOps.empty()) {
                compactOps.push_back(item);
            } else if (item < 0 && compactOps.back() < 0) {
                int lastMask = -compactOps.back();
                int nowMask = -item;
                int mergedMask = lastMask ^ nowMask;
                compactOps.back() = (mergedMask == 0 ? 0 : -mergedMask);
                if (compactOps.back() == 0) {
                    compactOps.pop_back();
                }
            } else {
                compactOps.push_back(item);
            }
        }
        return {true, compactOps};
    }
    bool alignBasePart() {
        if (segmentLen <= 1) {
            return true;
        }
        vector<int> lowBits(segmentLen);
        for (int idx = 0; idx < segmentLen; ++idx) {
            lowBits[idx] = currentSeq[idx] & (segmentLen - 1);
        }
        auto [valid, plan] = decomposeOrder(lowBits, segmentLen);
        if (!valid) {
            return false;
        }
        for (int item : plan) {
            if (item > 0) {
                useShift(item);
            } else {
                useXor(-item);
            }
        }
        return true;
    }
    bool makeIdentity() {
        for (int residue = 0; residue < segmentLen; ++residue) {
            vector<int> bucket;
            for (int pos = residue; pos < sizeN; pos += segmentLen) {
                bucket.push_back(currentSeq[pos]);
            }
            sort(bucket.begin(), bucket.end());
            int cursor = 0;
            for (int pos = residue; pos < sizeN; pos += segmentLen) {
                if (bucket[cursor] != pos) {
                    return false;
                }
                ++cursor;
            }
            for (int pos = residue; pos < sizeN; pos += segmentLen) {
                if (currentSeq[pos] != pos) {
                    exchangeValuePair(pos, currentSeq[pos]);
                }
            }
        }
        return true;
    }
    void outputCommands() {
        cout << commandList.size() << "\n";
        for (int item : commandList) {
            if (item == 0) {
                cout << "0\n";
            } else if (item < 0) {
                cout << "1 " << -item << "\n";
            } else {
                cout << "2 " << item << "\n";
            }
        }
    }
};
int main() {
    PermutationOperator task;
    task.solve();
    return 0;
}

## B 长跑

In [ ]:
#include <bits/stdc++.h>
using namespace std;
const int MAXN = 2005;
const int MAXS = 2005;
const int INF = -1e9;
struct Stop {
    int pos, cost;
} stops[MAXN];
int dp[MAXS];
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int N, L, Maxn, S;
    while (cin >> N >> L >> Maxn >> S) {
        for (int i = 0; i < N; ++i) {
            cin >> stops[i].pos >> stops[i].cost;
        }
        sort(stops, stops + N, [](const Stop& a, const Stop& b) {
            if (a.pos != b.pos) return a.pos < b.pos;
            return a.cost < b.cost;
        });
        vector<Stop> st;
        for (int i = 0; i < N; ++i) {
            if (i > 0 && stops[i].pos == stops[i - 1].pos) continue;
            st.push_back(stops[i]);
        }
        st.push_back({L, 0});
        int m = st.size();
        fill(dp, dp + S + 1, INF);
        dp[S] = Maxn;
        int prev_pos = 0;
        bool ok = false;
        for (int i = 0; i < m; ++i) {
            int curr_pos = st[i].pos;
            int curr_cost = st[i].cost;
            int dist = curr_pos - prev_pos;
            for (int j = 0; j <= S; ++j) {
                if (dp[j] >= dist) dp[j] -= dist;
                else dp[j] = INF;
            }
            if (curr_pos == L) {
                for (int j = 0; j <= S; ++j) {
                    if (dp[j] >= 0) {
                        ok = true;
                        break;
                    }
                }
                break;
            }
            for (int j = S; j >= curr_cost; --j) {
                if (dp[j] != INF) {
                    dp[j - curr_cost] = max(dp[j - curr_cost], Maxn);
                }
            }
            prev_pos = curr_pos;
        }
        cout << (ok ? "Yes" : "No") << '\n';
    }
    return 0;
}

## C 最长回文

In [ ]:
#include <bits/stdc++.h>
using namespace std;
using ll = long long;
static const ll MOD1 = 1000000007LL;
static const ll MOD2 = 1000000009LL;
static const ll BASE = 911382323LL;
struct DoubleHash {
    int n;
    vector<ll> h1, h2, p1, p2;
    string s;
    DoubleHash() {}
    DoubleHash(const string &str) {
        init(str);
    }
    void init(const string &str) {
        s = " " + str;
        n = (int)str.size();
        h1.assign(n + 1, 0);
        h2.assign(n + 1, 0);
        p1.assign(n + 1, 1);
        p2.assign(n + 1, 1);
        for (int i = 1; i <= n; i++) {
            int v = s[i] - 'A' + 1;
            h1[i] = (h1[i - 1] * BASE + v) % MOD1;
            h2[i] = (h2[i - 1] * BASE + v) % MOD2;
            p1[i] = (p1[i - 1] * BASE) % MOD1;
            p2[i] = (p2[i - 1] * BASE) % MOD2;
        }
    }
    pair<ll, ll> get(int l, int r) const {
        if (l > r) return {0, 0};
        ll x1 = (h1[r] - h1[l - 1] * p1[r - l + 1] % MOD1 + MOD1) % MOD1;
        ll x2 = (h2[r] - h2[l - 1] * p2[r - l + 1] % MOD2 + MOD2) % MOD2;
        return {x1, x2};
    }
};
void manacher(const string &s, vector<int> &d1, vector<int> &d2) {
    int n = (int)s.size();
    d1.assign(n, 0);
    d2.assign(n, 0);
    for (int i = 0, l = 0, r = -1; i < n; i++) {
        int k = (i > r ? 1 : min(d1[l + r - i], r - i + 1));
        while (0 <= i - k && i + k < n && s[i - k] == s[i + k]) k++;
        d1[i] = k;
        if (i + k - 1 > r) {
            l = i - k + 1;
            r = i + k - 1;
        }
    }
    for (int i = 0, l = 0, r = -1; i < n; i++) {
        int k = (i > r ? 0 : min(d2[l + r - i + 1], r - i + 1));
        while (0 <= i - k - 1 && i + k < n && s[i - k - 1] == s[i + k]) k++;
        d2[i] = k;
        if (i + k - 1 > r) {
            l = i - k;
            r = i + k - 1;
        }
    }
}
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int n;
    string A, B;
    cin >> n >> A >> B;
    string revA = A;
    reverse(revA.begin(), revA.end());
    DoubleHash hashRevA(revA), hashB(B);
    auto ext = [&](int p, int q) -> int {
        if (p <= 0 || q > n) return 0;
        int posRevA = n - p + 1;
        int maxLen = min(p, n - q + 1);
        int l = 0, r = maxLen;
        while (l < r) {
            int mid = (l + r + 1) >> 1;
            if (hashRevA.get(posRevA, posRevA + mid - 1) == hashB.get(q, q + mid - 1)) {
                l = mid;
            } else {
                r = mid - 1;
            }
        }
        return l;
    };
    vector<int> d1A, d2A, d1B, d2B;
    manacher(A, d1A, d2A);
    manacher(B, d1B, d2B);
    int ans = 0;
    for (int k = 1; k <= n; k++) {
        ans = max(ans, 2 * ext(k, k));
    }
    for (int i = 0; i < n; i++) {
        int c = i + 1;
        int rad = d1A[i];
        int l = c - rad + 1;
        int r = c + rad - 1;
        int cand = (2 * rad - 1) + 2 * ext(l - 1, r);
        ans = max(ans, cand);
    }
    for (int i = 0; i < n; i++) {
        int rad = d2A[i];
        if (rad == 0) continue;
        int c = i + 1;
        int l = c - rad;
        int r = c + rad - 1;
        int cand = (2 * rad) + 2 * ext(l - 1, r);
        ans = max(ans, cand);
    }
    for (int i = 0; i < n; i++) {
        int c = i + 1;
        int rad = d1B[i];
        int l = c - rad + 1;
        int r = c + rad - 1;
        int cand = (2 * rad - 1) + 2 * ext(l, r + 1);
        ans = max(ans, cand);
    }
    for (int i = 0; i < n; i++) {
        int rad = d2B[i];
        if (rad == 0) continue;
        int c = i + 1;
        int l = c - rad;
        int r = c + rad - 1;
        int cand = (2 * rad) + 2 * ext(l, r + 1);
        ans = max(ans, cand);
    }
    cout << ans << '\n';
    return 0;
}

## D 优惠券

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <set>
#include <algorithm>
using namespace std;
void solve() {
    int m;
    vector<int> balance(100005, 0);
    vector<int> last_op(100005, -1);
    vector<int> seen;
    while (cin >> m) {
        set<int> q_indices;
        int error_line = -1;
        for (int x : seen) {
            if (x < balance.size()) {
                balance[x] = 0;
                last_op[x] = -1;
            }
        }
        seen.clear();
        for (int i = 1; i <= m; ++i) {
            string op_str;
            cin >> op_str;
            if (error_line != -1) {
                if (op_str != "?") {
                    int x;
                    cin >> x;
                }
                continue;
            }
            if (op_str == "?") {
                q_indices.insert(i);
            } else {
                int x;
                cin >> x;
                if (x >= balance.size()) {
                    int new_size = max((int)balance.size() * 2, x + 1);
                    balance.resize(new_size, 0);
                    last_op.resize(new_size, -1);
                }
                if (last_op[x] == -1) {
                    seen.push_back(x);
                }
                char op = op_str[0];
                if (op == 'I' || op == 'i') {
                    if (balance[x] == 1) {
                        auto it = q_indices.upper_bound(last_op[x]);
                        if (it != q_indices.end()) {
                            q_indices.erase(it);
                            last_op[x] = i;
                        } else {
                            error_line = i;
                        }
                    } else {
                        balance[x] = 1;
                        last_op[x] = i;
                    }
                } else if (op == 'O' || op == 'o') {
                    if (balance[x] == 0) {
                        auto it = q_indices.upper_bound(last_op[x]);
                        if (it != q_indices.end()) {
                            q_indices.erase(it);
                            last_op[x] = i;
                        } else {
                            error_line = i;
                        }
                    } else {
                        balance[x] = 0;
                        last_op[x] = i;
                    }
                }
            }
        }
        cout << error_line << "\n";
    }
}
int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    solve();
    return 0;
}

## E 任意点

In [ ]:
#include <bits/stdc++.h>
using namespace std;
const int MAXN = 105;
const int MAXC = 1005;
struct DSU {
    vector<int> fa;
    DSU(int n) : fa(n + 1) {
        iota(fa.begin(), fa.end(), 0);
    }
    int find(int x) {
        return fa[x] == x ? x : fa[x] = find(fa[x]);
    }
    void unite(int x, int y) {
        x = find(x), y = find(y);
        if (x != y) fa[y] = x;
    }
};
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int n;
    cin >> n;
    vector<pair<int, int>> pts(n);
    for (int i = 0; i < n; ++i) {
        cin >> pts[i].first >> pts[i].second;
    }
    DSU dsu(2000);
    for (auto &p : pts) {
        int x = p.first;
        int y = p.second;
        dsu.unite(x, 1000 + y);
    }
    unordered_set<int> components;
    for (auto &p : pts) {
        int root = dsu.find(p.first);
        components.insert(root);
    }
    cout << components.size() - 1 << endl;
    return 0;
}

## F 通配符匹配

In [ ]:
#include <iostream>
#include <string>
#include <vector>
#include <cstring>
using namespace std;
const int MAX_LEN = 100005;
char V[MAX_LEN];
char next_V[MAX_LEN];
char match_arr[MAX_LEN];
struct Part {
    char w;
    string s;
    vector<int> pi;
};
vector<int> get_pi(const string& s) {
    int m = s.length();
    vector<int> pi(m);
    for (int i = 1, j = 0; i < m; i++) {
        while (j > 0 && s[i] != s[j]) j = pi[j - 1];
        if (s[i] == s[j]) j++;
        pi[i] = j;
    }
    return pi;
}
void get_matches(const string& S, const string& W, const vector<int>& pi, char match[]) {
    int S_len = S.length();
    int W_len = W.length();
    memset(match, 0, S_len + 1);
    if (W_len == 0) {
        memset(match, 1, S_len + 1);
        return;
    }
    for (int i = 0, j = 0; i < S_len; i++) {
        while (j > 0 && S[i] != W[j]) j = pi[j - 1];
        if (S[i] == W[j]) j++;
        if (j == W_len) {
            match[i - W_len + 1] = 1;
            j = pi[j - 1];
        }
    }
}
int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(NULL);
    string P;
    if (!(cin >> P)) return 0;
    int n;
    cin >> n;
    string S0 = "";
    vector<Part> parts;
    int cur = 0;
    while (cur < P.length() && P[cur] != '*' && P[cur] != '?') {
        S0 += P[cur];
        cur++;
    }
    while (cur < P.length()) {
        Part p;
        p.w = P[cur];
        cur++;
        while (cur < P.length() && P[cur] != '*' && P[cur] != '?') {
            p.s += P[cur];
            cur++;
        }
        p.pi = get_pi(p.s);
        parts.push_back(p);
    }
    while (n--) {
        string S;
        cin >> S;
        int S_len = S.length();
        if (S_len < S0.length()) {
            cout << "NO\n";
            continue;
        }
        bool match_S0 = true;
        for (int i = 0; i < S0.length(); i++) {
            if (S[i] != S0[i]) {
                match_S0 = false;
                break;
            }
        }
        if (!match_S0) {
            cout << "NO\n";
            continue;
        }
        memset(V, 0, S_len + 1);
        V[S0.length()] = 1;
        bool possible = true;
        for (int i = 0; i < parts.size(); i++) {
            get_matches(S, parts[i].s, parts[i].pi, match_arr);
            memset(next_V, 0, S_len + 1);
            int W_len = parts[i].s.length();
            if (parts[i].w == '?') {
                for (int j = 0; j <= S_len - 1 - W_len; j++) {
                    if (V[j] && match_arr[j + 1]) {
                        next_V[j + 1 + W_len] = 1;
                    }
                }
            } else {
                int min_j = -1;
                for (int j = 0; j <= S_len; j++) {
                    if (V[j]) {
                        min_j = j;
                        break;
                    }
                }
                if (min_j != -1) {
                    for (int k = min_j; k <= S_len - W_len; k++) {
                        if (match_arr[k]) {
                            next_V[k + W_len] = 1;
                        }
                    }
                }
            }
            bool any_true = false;
            for (int j = 0; j <= S_len; j++) {
                if (next_V[j]) {
                    any_true = true;
                    break;
                }
            }
            memcpy(V, next_V, S_len + 1);
            if (!any_true) {
                possible = false;
                break;
            }
        }
        if (possible && V[S_len]) {
            cout << "YES\n";
        } else {
            cout << "NO\n";
        }
    }
    return 0;
}

## G 汉诺塔

In [ ]:
#include <bits/stdc++.h>
using namespace std;
int id(char c) {
    return c - 'A';
}
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int n;
    cin >> n;
    vector<pair<int, int>> order(6);
    for (int i = 0; i < 6; ++i) {
        string s;
        cin >> s;
        order[i] = {id(s[0]), id(s[1])};
    }
    vector<vector<long long>> dp(n + 1, vector<long long>(3, 0));
    vector<vector<int>> to(n + 1, vector<int>(3, 0));
    for (int s = 0; s < 3; ++s) {
        for (auto mv : order) {
            if (mv.first == s) {
                to[1][s] = mv.second;
                dp[1][s] = 1;
                break;
            }
        }
    }
    for (int k = 2; k <= n; ++k) {
        for (int s = 0; s < 3; ++s) {
            int small = s;
            int large = s;
            long long cnt = 0;
            while (true) {
                cnt += dp[k - 1][small];
                small = to[k - 1][small];
                if (small == large) {
                    dp[k][s] = cnt;
                    to[k][s] = large;
                    break;
                }
                int empty = 3 - small - large;
                large = empty;
                cnt++;
            }
        }
    }
    cout << dp[n][0] << '\n';
    return 0;
}

## H 马步距离

In [ ]:
#include <iostream>
#include <cmath>
#include <algorithm>
using namespace std;
int main() {
    long long xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;
    long long dx = abs(xp - xs);
    long long dy = abs(yp - ys);
    if (dx < dy) swap(dx, dy);
    long long ans;
    if (dx == 0 && dy == 0) {
        ans = 0;
    } else if (dx == 1 && dy == 0) {
        ans = 3;
    } else if (dx == 2 && dy == 2) {
        ans = 4;
    } else {
        ans = max({
            (dx + 1) / 2,
            (dx + dy + 2) / 3
        });
        while ((ans - dx - dy) % 2 != 0) {
            ans++;
        }
    }
    cout << ans << endl;
    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution {
public:
    int largestRectangleArea(vector<int>& heights) {
        stack<int> st;
        heights.push_back(0);
        int max_area = 0;
        for (int i = 0; i < heights.size(); ++i) {
            while (!st.empty() && heights[i] < heights[st.top()]) {
                int h = heights[st.top()];
                st.pop();
                int w = st.empty() ? i : i - st.top() - 1;
                max_area = max(max_area, h * w);
            }
            st.push(i);
        }
        return max_area;
    }
};

## J 消防局的设立

In [ ]:
#include <bits/stdc++.h>
using namespace std;
int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int n;
    cin >> n;
    const int INF = 1e9;
    vector<int> fa(n + 1, 0);
    for (int i = 2; i <= n; ++i) {
        cin >> fa[i];
    }
    vector<int> mn(n + 1, INF);
    vector<int> req(n + 1, -1);
    vector<int> dist(n + 1, INF);
    vector<char> hasReq0(n + 1, 0), hasReq1(n + 1, 0);
    int ans = 0;
    for (int u = n; u >= 1; --u) {
        if (hasReq1[u]) {
            ++ans;
            req[u] = -1;
            dist[u] = 0;
        } else {
            bool unresolvedChild = hasReq0[u] && (mn[u] != 1);
            if (unresolvedChild) {
                req[u] = 1;
                dist[u] = mn[u];
            } else if (mn[u] <= 2) {
                req[u] = -1;
                dist[u] = mn[u];
            } else {
                req[u] = 0;
                dist[u] = INF;
            }
        }
        if (u > 1) {
            int p = fa[u];
            if (req[u] == 0) {
                hasReq0[p] = 1;
            } else if (req[u] == 1) {
                hasReq1[p] = 1;
            }
            if (dist[u] < INF) {
                mn[p] = min(mn[p], dist[u] + 1);
            }
        }
    }
    if (req[1] != -1 || dist[1] > 2) {
        ++ans;
    }
    cout << ans << '\n';
    return 0;
}